# 06. Training Fine-Tuning and Alignment

Training and alignment pipelines are among the most important interview topics for LLM engineer roles because they connect model theory to production decision-making. A strong answer should explain *why* modern LLM development is multi-stage, what each stage optimizes, and where each method can fail.

This notebook provides a deeper, interview-focused treatment of pre-training, SFT, RLHF, DPO, LoRA/QLoRA, and full fine-tuning trade-offs. It also includes a concrete decision framework for when to use prompting, RAG, or fine-tuning; data requirements for adaptation; and practical risk-management guidance.

---


## 1. Why Training Is Multi-Stage

A single training objective cannot capture every requirement we care about in deployed systems. During pre-training, the model learns broad language structure and world patterns from massive text corpora, but it is not yet explicitly optimized for instruction-following, helpfulness, or policy constraints.

Later stages progressively shape behavior: supervised instruction data teaches response format, preference data teaches style and quality trade-offs, and alignment methods constrain unsafe or unhelpful behavior. Thinking in stages helps separate capability building from behavior steering.

---


## 2. Pre-Training: Base Capability Acquisition

Pre-training uses next-token prediction over diverse corpora. The model learns grammar, idioms, style transfer, latent reasoning patterns, and broad factual priors. This stage dominates compute cost and parameter learning.

Even very strong base models can still produce brittle instruction behavior because they optimize likelihood, not task success against human intent. As a result, pre-training is necessary but not sufficient for reliable assistant behavior.

---


## 3. Supervised Fine-Tuning (SFT)

SFT teaches the model to map prompts to desirable responses using curated demonstrations. It improves instruction-following, tone consistency, and response structure. In practice, SFT data quality matters more than raw quantity because the model quickly absorbs annotation style.

SFT can significantly improve task completion rates for domain-specific workflows, but it can also overfit formatting habits or inject dataset biases. Good SFT pipelines therefore include stratified sampling, style diversity, and explicit safety coverage.

---


## 4. RLHF: Preference Learning with Rewards

Reinforcement Learning from Human Feedback typically combines three parts: (1) collect ranked outputs from humans, (2) train a reward model to predict those preferences, and (3) optimize the policy model against that reward while limiting divergence from the SFT model.

RLHF can improve helpfulness and refusal behavior, but it introduces reward-model dependence. If reward modeling is misaligned with real user value, the policy can optimize superficial proxies. Monitoring for reward hacking and maintaining high-quality preference data is therefore essential.

---


## 5. DPO: Direct Preference Optimization

DPO reframes preference optimization without an explicit reward model rollout loop. It learns directly from preferred vs rejected response pairs by increasing likelihood of preferred answers relative to non-preferred ones.

Compared with full RLHF pipelines, DPO is often simpler operationally and easier to stabilize. The trade-off is that it still relies on preference dataset quality and may not capture long-horizon interaction effects that iterative policy optimization can target.

---


## 6. LoRA and PEFT for Efficient Adaptation

Parameter-Efficient Fine-Tuning (PEFT) methods avoid updating all model weights. LoRA, for example, injects low-rank trainable matrices into selected layers while keeping the base model frozen. This greatly reduces trainable parameters, GPU memory, and training cost.

PEFT is especially useful when teams need multiple domain variants of a common base model. Instead of storing full model copies, teams store small adapters and swap them depending on the task context.

---


In [ ]:
import numpy as np

# Simple parameter count comparison for full fine-tuning vs LoRA-style adapters
model_params = np.array([7e9, 13e9, 70e9])
lora_fraction = 0.0025  # 0.25% trainable parameters as an illustrative value

full_trainable = model_params
lora_trainable = model_params * lora_fraction

for total, full, lora in zip(model_params, full_trainable, lora_trainable):
    print(f"Model: {total/1e9:.0f}B params | Full FT: {full/1e9:.2f}B | LoRA: {lora/1e6:.2f}M")

reduction = full_trainable / lora_trainable
print("\nApproximate trainable-parameter reduction factor:", reduction)


## 7. Training Stages Diagram

A useful mental model is to treat each stage as shaping a different axis of behavior:

```text
Raw Text + Code Corpora
          |
          v
   [Pre-Training]
  Learn broad language priors
          |
          v
 [Supervised Fine-Tuning]
 Learn instruction formatting and task behavior
          |
          +-----------------------------+
          |                             |
          v                             v
      [RLHF]                         [DPO]
 Reward-driven policy shaping     Pairwise preference shaping
          \                             /
           \                           /
            v                         v
         [Aligned Assistant for Deployment]
```

This pipeline is iterative in practice: deployment feedback often leads to additional SFT/preference data and periodic refreshes.

---


## 8. Prompt vs Fine-Tune vs RAG: Decision Framework

Use prompting first when task behavior can be specified clearly with instructions/examples and quality is already acceptable. Prompt iteration is fastest, cheapest, and lowest risk.

Use RAG when failures are mainly about missing or stale knowledge. Retrieval injects grounded context without changing model weights. Use fine-tuning when failures are about behavior patterns, tone consistency, output schema, or domain-specific writing style not fixed by prompt design.

A practical order is: Prompt optimization -> RAG for knowledge grounding -> Fine-tuning for persistent behavior gaps.

---


In [ ]:
import numpy as np

# Toy scoring rubric for choosing strategy
# Columns: knowledge_gap, style_gap, format_consistency_need, latency_budget_tight
scenarios = {
    "FAQ assistant with evolving policies": np.array([0.9, 0.3, 0.4, 0.5]),
    "Strict JSON extraction at scale": np.array([0.2, 0.6, 0.9, 0.8]),
    "General brainstorming helper": np.array([0.3, 0.2, 0.2, 0.4]),
}

for name, v in scenarios.items():
    knowledge_gap, style_gap, format_need, latency_tight = v
    prompt_score = (1 - format_need) + (1 - style_gap)
    rag_score = knowledge_gap + 0.3 * (1 - latency_tight)
    finetune_score = style_gap + format_need + 0.2 * latency_tight
    scores = {"prompt": prompt_score, "rag": rag_score, "fine_tune": finetune_score}
    best = max(scores, key=scores.get)
    print(f"{name}: {scores} -> recommended: {best}")


## 9. Data Curation and Alignment Quality

Alignment quality follows dataset quality. Preference labels should reflect real user value, not just superficial politeness. For high-stakes domains, annotation guidelines should explicitly encode uncertainty handling, citation expectations, and abstention criteria.

Distribution coverage matters: if edge cases are absent from SFT and preference data, models appear strong in demos but fail in production tails. Representative sampling and hard-case mining are therefore central to robust alignment.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate performance gain over stages on an internal benchmark
stages = ["Base", "SFT", "DPO/RLHF", "Domain Adapter"]
scores = np.array([0.52, 0.69, 0.78, 0.84])

plt.figure(figsize=(8, 4))
plt.plot(stages, scores, marker="o")
plt.ylim(0.4, 0.9)
plt.ylabel("Task Success Rate")
plt.title("Illustrative Improvement Across Training Stages")
plt.grid(alpha=0.3)
plt.show()


## 10. Risks and Trade-Offs

Fine-tuning can improve consistency but may reduce generality if domain data is narrow. Preference optimization can improve UX while hiding errors behind confident style. PEFT reduces cost but still inherits base-model limitations.

The right pipeline balances quality gain, operational complexity, latency, and governance requirements. In many systems, hybrid strategies are best: light fine-tuning for format/style plus RAG for fresh factual grounding.

---


## 11. Practical Implementation Checklist

Define one measurable target before training (for example, schema-valid completion rate or grounded answer accuracy). Build a compact but representative evaluation set. Start with prompt and retrieval baselines, then justify training cost with observed residual failure categories.

For adaptation, begin with PEFT/LoRA unless full fine-tuning is clearly necessary. Monitor regressions continuously and keep rollback-ready checkpoints. Alignment should be treated as an ongoing process rather than a one-time step.

---


## 12. Summary

Pre-training builds broad capability, SFT teaches instruction behavior, and RLHF/DPO shape preferences and alignment. LoRA/PEFT provides a cost-efficient path to domain adaptation. For real systems, choose among prompting, RAG, and fine-tuning based on whether the core gap is instruction clarity, missing knowledge, or persistent behavior mismatch.

A robust strategy is iterative: measure failures, select the minimal intervention that closes them, and reevaluate after deployment feedback.


---

## 13. Step-by-Step: RLHF Pipeline Breakdown

In interviews, explain RLHF as an operational sequence rather than a buzzword:

1. **Collect prompts and candidate responses** (from model checkpoints or sampling policies).  
2. **Rank responses with human preference labels** (or high-quality proxy annotators).  
3. **Train a reward model** to predict preference scores.  
4. **Optimize the policy model** (e.g., PPO-style) to maximize reward while constraining divergence from SFT policy.  
5. **Run safety and regression evaluation** before deployment.

A common objective includes a KL regularization term:

\[
\max_{\pi} \; \mathbb{E}_{y\sim\pi(\cdot|x)}[r_\phi(x,y)] - \beta \cdot D_{KL}(\pi(\cdot|x)\,\|\,\pi_{\text{SFT}}(\cdot|x))
\]

where \(\beta\) controls how far the policy can drift from the supervised model. Mentioning KL control is often viewed positively in interviews because it shows practical alignment intuition.

---

## 14. DPO Objective Intuition

DPO uses pairwise preference data \((y^+, y^-)\) and directly optimizes relative likelihood without separate reward-model rollouts. Intuitively, it increases probability of preferred outputs while decreasing rejected outputs under a reference-model-anchored objective.

A common form is:

\[
\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \left[\log \frac{\pi_\theta(y^+|x)}{\pi_{\text{ref}}(y^+|x)} - \log \frac{\pi_\theta(y^-|x)}{\pi_{\text{ref}}(y^-|x)}\right]\right)
\]

DPO is operationally simpler than full RLHF loops, but interviewers appreciate hearing its limits: quality still depends on preference dataset coverage, and long-horizon conversational behavior may require broader evaluation than pairwise training signals alone.

---

## 15. LoRA vs QLoRA vs Full Fine-Tuning

| Method | Trainable Weights | Hardware Cost | Quality Ceiling | Typical Use Case |
|---|---|---|---|---|
| Full fine-tuning | Most/all parameters | Highest | Potentially highest with enough data | Large orgs, high-volume domain specialization |
| LoRA | Low-rank adapter params | Much lower | Often strong for format/style/domain behavior | Fast domain adaptation with limited budget |
| QLoRA | LoRA on quantized base | Lowest among three | Strong for many tasks, verify quality edge cases | Memory-constrained adaptation workflows |

Interview framing: start with PEFT (LoRA/QLoRA) unless your eval gap remains large and clearly justifies full fine-tuning complexity. Mention checkpoint management, rollback safety, and reproducibility tracking as production concerns beyond raw model quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrative trainable-parameter fractions for adaptation methods
base_params_b = 13.0  # 13B base model (illustrative)
ranks = np.array([4, 8, 16, 32, 64])

# Toy approximation: trainable fraction grows roughly linearly with rank
lora_fraction = 0.00012 * ranks  # illustrative, not architecture-accurate
qlora_fraction = 0.00009 * ranks
full_fraction = np.ones_like(ranks)

plt.figure(figsize=(8.5, 4.5))
plt.plot(ranks, lora_fraction * 100, marker="o", label="LoRA trainable %")
plt.plot(ranks, qlora_fraction * 100, marker="o", label="QLoRA trainable %")
plt.axhline(100, linestyle="--", label="Full fine-tuning")
plt.xlabel("Adapter rank (r)")
plt.ylabel("Trainable parameters (%)")
plt.title("Why PEFT is compute efficient")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for r, lf, qf in zip(ranks, lora_fraction, qlora_fraction):
    print(f"r={r:>2} | LoRA trainable ~{lf*base_params_b*1e3:6.2f}M | QLoRA ~{qf*base_params_b*1e3:6.2f}M")

---

## 16. Prompt vs RAG vs Fine-Tune (Decision Tree)

Use this practical decision tree in interviews:

- If failures are mostly due to unclear instructions -> improve prompting/template design first.  
- If failures are mostly due to missing/stale facts -> use RAG and retrieval quality controls.  
- If failures persist as behavior/style/schema consistency gaps -> use fine-tuning (prefer LoRA/QLoRA first).

\[
\text{Intervention Priority} = \arg\max\{S_{\text{prompt}}, S_{\text{RAG}}, S_{\text{FT}}\}
\]

Where each score is computed from measured failure categories (knowledge gap, formatting gap, domain language mismatch, latency budget, and governance constraints). The key interview point: choose the *minimal intervention* that closes the dominant failure mode.

---

## 17. Data Requirements for Fine-Tuning

Data quality dominates fine-tuning outcomes. High-quality, diverse, policy-consistent examples usually outperform larger but noisy datasets. For interview answers, discuss both quantity and coverage:

| Data Aspect | Why It Matters | Practical Guidance |
|---|---|---|
| Task coverage | Prevents brittle behavior | Include easy, medium, hard, and adversarial cases |
| Label quality | Determines learned behavior | Strong annotation rubric + multi-pass QA |
| Format consistency | Improves schema reliability | Standardize output templates and edge-case handling |
| Safety coverage | Reduces harmful failure modes | Add refusal, uncertainty, and escalation examples |
| Evaluation split | Detects overfitting/regression | Keep held-out test sets by scenario and risk level |

Roughly, many teams begin with a few thousand high-quality examples for narrow behavior adaptation, then scale only if measured gaps remain. Explicitly say "we scale data based on eval deltas" to show mature experimentation discipline.

In [ ]:
import numpy as np

# Toy decision score grid for Prompt vs RAG vs Fine-tune
# Inputs scaled in [0, 1]
knowledge_gap = np.array([0.2, 0.5, 0.8])
behavior_gap = np.array([0.2, 0.5, 0.8])

print("knowledge_gap | behavior_gap -> best strategy")
for kg in knowledge_gap:
    for bg in behavior_gap:
        prompt_score = (1 - kg) * 0.6 + (1 - bg) * 0.8
        rag_score = kg * 1.0 + (1 - bg) * 0.2
        ft_score = bg * 1.0 + 0.2 * (1 - kg)
        scores = {"prompt": prompt_score, "rag": rag_score, "fine_tune": ft_score}
        best = max(scores, key=scores.get)
        print(f"    {kg:.1f}       |     {bg:.1f}     -> {best:10s} {scores}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrative training/validation curves across stages
steps = np.arange(1, 31)
pretrain_loss = 2.5 / np.sqrt(steps) + 0.25
sft_loss = 1.4 / np.sqrt(steps) + 0.18
pref_opt_loss = 1.2 / np.sqrt(steps) + 0.15

plt.figure(figsize=(9, 4.5))
plt.plot(steps, pretrain_loss, label="Pre-training objective")
plt.plot(steps, sft_loss, label="SFT objective")
plt.plot(steps, pref_opt_loss, label="Preference optimization objective")
plt.xlabel("Training step (normalized)")
plt.ylabel("Illustrative loss")
plt.title("Different stages optimize different targets")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---

## 18. Interview Focus (Detailed Q&A)

**Q1. What is RLHF in simple terms?**  
RLHF uses human preferences to train a reward signal and then optimizes the model to produce outputs that score higher while keeping behavior near the SFT baseline.

**Q2. RLHF vs DPO?**  
RLHF uses reward modeling plus policy optimization loops; DPO optimizes pairwise preference data directly and is often operationally simpler.

**Q3. LoRA vs full fine-tuning?**  
LoRA updates a small adapter subset, reducing cost and memory dramatically. Full FT offers maximum flexibility but requires much more compute/data and stronger MLOps discipline.

**Q4. What is QLoRA?**  
QLoRA combines quantized base weights with LoRA adapters, enabling large-model adaptation in limited-memory environments.

**Q5. When should you fine-tune instead of using prompting?**  
When failure modes are persistent behavior/style/schema issues that prompt engineering cannot reliably fix.

**Q6. When is RAG better than fine-tuning?**  
When the dominant issue is missing or stale factual knowledge rather than behavioral consistency.

**Q7. How much data is needed for fine-tuning?**  
It depends on task complexity and quality requirements, but high-quality curated examples with broad edge-case coverage matter more than raw count.

**Q8. Biggest alignment risk?**  
Optimizing proxy preference metrics that look good offline but fail in real user scenarios due to distribution mismatch.

**Q9. How do you evaluate post-training improvements?**  
Track task success, safety metrics, schema validity, and regression slices against a fixed baseline with statistically meaningful comparisons.

**Q10. What is a strong practical strategy?**  
Prompt baseline -> RAG for knowledge grounding -> PEFT fine-tuning for persistent behavior gaps -> preference alignment with robust evaluation loops.

---

## 19. Summary for Interviews

Pre-training builds broad capabilities, SFT teaches instruction behavior, and preference optimization (RLHF/DPO) aligns response quality with human expectations. LoRA/QLoRA offer efficient adaptation paths that are often the best first choice before full fine-tuning. The best production decisions come from measured failure analysis: use prompting for instruction gaps, RAG for knowledge gaps, and fine-tuning for persistent behavioral gaps.

## Key Takeaways (Interview-Ready)

- Explain LLM training as staged optimization with distinct objectives.
- RLHF and DPO both use preference data, but differ in optimization mechanics.
- Start adaptation with PEFT unless evaluation evidence justifies full fine-tuning.
- Fine-tuning data quality/coverage is usually more important than raw volume.
- Choose prompt vs RAG vs fine-tune using failure-category evidence, not intuition.